In [1]:
INTERVAL = "1h"
DAYS     = 365
N_COINS  = 10
QUOTE    = "USDT"
RAW_DIR  = "raw"
EXCLUDE  = ("USDC", "FDUSD", "TUSD", "BUSD", "DAI",  # stable/stable pairs
            "UP", "DOWN", "BULL", "BEAR")            # leveraged tokens

In [2]:
import time, requests, pandas as pd
from pathlib import Path

SPOT_KL  = "https://api.binance.com/api/v3/klines"
SPOT_24H = "https://api.binance.com/api/v3/ticker/24hr"
FUT_KL   = "https://fapi.binance.com/fapi/v1/klines"
FUT_INFO = "https://fapi.binance.com/fapi/v1/exchangeInfo"
Path(RAW_DIR).mkdir(exist_ok=True)

def get(url, **params):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    time.sleep(0.25)                 # stay well below the rate limits
    return r.json()

end_ms   = int(time.time() * 1000)
start_ms = end_ms - DAYS * 86_400_000
print(pd.to_datetime(start_ms, unit="ms"), "->",
      pd.to_datetime(end_ms, unit="ms"))

2025-08-28 14:25:48.352000 -> 2026-08-28 14:25:48.352000


In [3]:
tick = pd.DataFrame(get(SPOT_24H))
tick["quoteVolume"] = tick.quoteVolume.astype(float)
perp = {s["symbol"] for s in get(FUT_INFO)["symbols"]
        if s.get("contractType") == "PERPETUAL" and s["status"] == "TRADING"}

cand = (tick[tick.symbol.str.endswith(QUOTE)
             & ~tick.symbol.str.contains("|".join(EXCLUDE))
             & tick.symbol.isin(perp)]
        .sort_values("quoteVolume", ascending=False)
        .symbol.head(3 * N_COINS).tolist())
print(len(cand), "candidates:", cand[:15])

30 candidates: ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT', 'ENAUSDT', 'ZECUSDT', 'TRUMPUSDT', 'DOGEUSDT', 'SUIUSDT', 'PUMPUSDT', 'TRXUSDT', 'LINKUSDT', 'HEMIUSDT', 'TAOUSDT', 'UNIUSDT']


## Download helper (paginated, stagnation-safe)

In [4]:
COLS = ["open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "n_trades",
        "taker_buy_base", "taker_buy_quote", "ignore"]

def fetch_klines(url, symbol, interval, t0, t1):
    rows, t = [], t0
    while t < t1:
        batch = get(url, symbol=symbol, interval=interval,
                    startTime=t, endTime=t1, limit=1000)
        if not batch:
            break
        rows += batch
        t_new = batch[-1][0] + 1
        if t_new <= t:               # never spin on one page
            break
        t = t_new
    if not rows:
        return pd.DataFrame(columns=COLS)
    return (pd.DataFrame(rows, columns=COLS)
              .drop_duplicates("open_time").sort_values("open_time"))

## Fetch spot, rank by realized volume, keep the top 10

One pass over the candidates: download the spot year, measure realized
quote volume and coverage, then keep the best `N_COINS` that have a
reasonably complete record.

In [5]:
expected = DAYS * 24 if INTERVAL == "1h" else None
spot, stats = {}, []
for sym in cand:
    kl = fetch_klines(SPOT_KL, sym, INTERVAL, start_ms, end_ms)
    if len(kl) == 0:
        continue
    spot[sym] = kl
    stats.append({"symbol": sym,
                  "bars": len(kl),
                  "coverage": len(kl) / expected if expected else 1.0,
                  "realized_quote_vol": kl.quote_volume.astype(float).sum()})
    print(f"\r{sym}: {len(kl)} bars", end="")
stats = pd.DataFrame(stats)
top = (stats[stats.coverage > 0.95]
       .sort_values("realized_quote_vol", ascending=False)
       .head(N_COINS))
print("\n\nselected:\n", top[["symbol", "bars", "coverage"]].to_string(index=False))

XLMUSDT: 8760 barsrs

selected:
   symbol  bars  coverage
 BTCUSDT  8760       1.0
 ETHUSDT  8760       1.0
 SOLUSDT  8760       1.0
 XRPUSDT  8760       1.0
 ZECUSDT  8760       1.0
DOGEUSDT  8760       1.0
 SUIUSDT  8760       1.0
 TRXUSDT  8760       1.0
 ADAUSDT  8760       1.0
PAXGUSDT  8760       1.0


## Save spot + perp klines for the selected coins

In [6]:
for sym in top.symbol:
    spot[sym].to_csv(f"{RAW_DIR}/spot_{sym}_{INTERVAL}.csv", index=False)
    fu = fetch_klines(FUT_KL, sym, INTERVAL, start_ms, end_ms)
    fu.to_csv(f"{RAW_DIR}/perp_{sym}_{INTERVAL}.csv", index=False)
    print(f"{sym}: spot {len(spot[sym])}, perp {len(fu)}")

top.to_csv(f"{RAW_DIR}/universe_{INTERVAL}.csv", index=False)
print("\nsaved to", RAW_DIR)

BTCUSDT: spot 8760, perp 8760
ETHUSDT: spot 8760, perp 8760
SOLUSDT: spot 8760, perp 8760
XRPUSDT: spot 8760, perp 8760
ZECUSDT: spot 8760, perp 8760
DOGEUSDT: spot 8760, perp 8760
SUIUSDT: spot 8760, perp 8760
TRXUSDT: spot 8760, perp 8760
ADAUSDT: spot 8760, perp 8760
PAXGUSDT: spot 8760, perp 8760

saved to raw
